# 🎬 影片使用情境分析器（Colab 批次版）

**功能：**
- 遞迴掃描 Google Drive 上的專案資料夾（含 `*_content.txt` 即視為專案）
- 從每個專案的 `*_resource_log.csv` 讀取影片 URL
- 逐一下載並分析每支影片的 `usage_scene_ratio`（使用情境畫面佔比）
- 輸出彙整 CSV

**輸出欄位：**
`project, category, video_filename, url, usage_scene_ratio, total_duration, sampled_frames, usage_frames, classifier_mode, status`

**資料夾結構：**
```
ZecZec_Dataset_E&G/
└── 預購式專案/教育/成功/
    └── ES32_hohoka-01/          ← 含 *_content.txt → 視為專案
        ├── ES32_resource_log.csv
        ├── ES32_content.txt
        ├── ES32_images/
        └── ES32_videos/
```

## ① 安裝依賴套件

In [ ]:
# 基本套件
!pip install -q opencv-python-headless yt-dlp gdown

# ✏️ 若要使用 CLIP 模式（較精準，需較長安裝時間），取消下行註解
# !pip install -q torch torchvision clip-by-openai

# OCR 支援（用於文字／字幕中的操作情境關鍵字偵測）
!apt-get install -q tesseract-ocr tesseract-ocr-chi-tra tesseract-ocr-chi-sim
!pip install -q pytesseract pillow

print('✅ 套件安裝完成')

## ② 掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive 已掛載')

## ③ 設定路徑與模式（依實際情況修改）

In [ ]:
import os

# ✏️ 根資料夾路徑
ROOT_DIR = '/content/drive/MyDrive/ZecZec_Group_Data/ZecZec_Dataset_E&G'

# ✏️ 輸出 CSV 路徑
OUTPUT_CSV = '/content/drive/MyDrive/ZecZec_Group_Data/video_features_result.csv'


# ✏️ 分類模式：'rule'（輕量，無需 GPU）或 'clip'（精準，需安裝 torch）
CLASSIFIER_MODE = 'rule'

# ✏️ 取幀間隔秒數（越小越精準但越慢）
SAMPLE_INTERVAL_SEC = 2.0

# ✏️ 暫存影片目錄（下載後自動清除）
TMP_DIR = '/content/tmp_videos'

print(f'根資料夾   : {ROOT_DIR}')
print(f'根資料夾存在: {os.path.isdir(ROOT_DIR)}')
print(f'輸出路徑   : {OUTPUT_CSV}')
print(f'分類模式   : {CLASSIFIER_MODE}')

## ④ 載入核心函式

In [ ]:
import re
import csv
import math
import tempfile
import logging
import subprocess
from enum import Enum
from dataclasses import dataclass
from typing import Optional

import cv2
import numpy as np

logging.basicConfig(level=logging.WARNING)  # Colab 內只顯示警告以上
logger = logging.getLogger(__name__)

# ── CLIP 可選依賴 ──────────────────────────────────────────
try:
    import torch
    import clip as openai_clip
    HAS_CLIP = True
except ImportError:
    HAS_CLIP = False

# ── OCR 可選依賴 ───────────────────────────────────────────
try:
    import pytesseract
    from PIL import Image as _PILImage
    HAS_TESSERACT = True
except ImportError:
    HAS_TESSERACT = False


# ══════════════════════════════════════════════════════════
# 設定
# ══════════════════════════════════════════════════════════
class ClassifierMode(Enum):
    CLIP = 'clip'
    RULE = 'rule'

USAGE_SCENE_PROMPTS = [
    'a person using the product',
    'a person operating a device',
    'a product demonstration in real life',
    'someone holding and using the product',
    'the product being used in everyday life',
]
NON_USAGE_SCENE_PROMPTS = [
    'a product on a white background',
    'a product specification sheet',
    'text and graphics on screen',
    'an animated infographic',
    'a logo or brand identity',
    'talking head interview',
]
RULE_SKIN_RATIO_THRESHOLD   = 0.04
RULE_EDGE_DENSITY_THRESHOLD = 0.08
RULE_BRIGHTNESS_MAX         = 230

# ── 操作情境關鍵字（中文 + 英文）──────────────────────────
# ✏️ 可依需求自行新增關鍵字
USAGE_KEYWORDS = [
    # 中文
    '操作', '使用', '示範', '教學', '步驟', '說明',
    '如何使用', '使用方式', '操作說明', '功能介紹',
    '安裝', '設定', '操作流程', '使用教學', '教程',
    '實際操作', '操作示範', '使用示範', '情境',
    # 英文
    'how to use', 'tutorial', 'demo', 'demonstration',
    'step', 'instruction', 'setup', 'install', 'guide',
    'operation', 'usage', 'how-to', 'walkthrough',
]


# ══════════════════════════════════════════════════════════
# OCR 關鍵字偵測（文字／字幕）
# ══════════════════════════════════════════════════════════
def _ocr_has_usage_keyword(frame_bgr: np.ndarray) -> bool:
    """
    對單一幀做 OCR，若辨識出的文字包含任一操作情境關鍵字回傳 True。
    Tesseract 未安裝時自動跳過，回傳 False。
    支援繁體中文、簡體中文、英文。
    """
    if not HAS_TESSERACT:
        return False
    try:
        rgb     = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pil_img = _PILImage.fromarray(rgb)
        # --psm 3: 全頁自動分析；lang 同時涵蓋繁中、簡中、英文
        text = pytesseract.image_to_string(
            pil_img,
            lang='chi_tra+chi_sim+eng',
            config='--psm 3'
        ).lower()
        return any(kw.lower() in text for kw in USAGE_KEYWORDS)
    except Exception as e:
        logger.warning(f'OCR 失敗: {e}')
        return False


# ══════════════════════════════════════════════════════════
# YouTube 下載
# ══════════════════════════════════════════════════════════
def extract_youtube_id(url: str) -> Optional[str]:
    patterns = [
        r'(?:v=|youtu\.be/|embed/)([A-Za-z0-9_-]{11})',
        r'shorts/([A-Za-z0-9_-]{11})',
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    return None

def download_youtube_video(url: str, output_dir: str, max_height: int = 360) -> Optional[str]:
    output_template = os.path.join(output_dir, '%(id)s.%(ext)s')
    cmd = [
        'yt-dlp',
        '--format', f'bestvideo[height<={max_height}][ext=mp4]+bestaudio[ext=m4a]/best[height<={max_height}][ext=mp4]/best',
        '--output', output_template,
        '--no-playlist', '--quiet', url,
    ]
    try:
        subprocess.run(cmd, check=True, timeout=180)
        for f in os.listdir(output_dir):
            if f.endswith(('.mp4', '.webm', '.mkv')):
                return os.path.join(output_dir, f)
    except Exception as e:
        logger.warning(f'yt-dlp 失敗: {e}')
    return None


# ══════════════════════════════════════════════════════════
# 幀取樣
# ══════════════════════════════════════════════════════════
def sample_frames(video_path: str, interval_sec: float) -> tuple[list[np.ndarray], float]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f'無法開啟影片: {video_path}')
    fps            = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total_duration = total_frames / fps
    step           = max(1, int(fps * interval_sec))
    frames, idx    = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            frames.append(frame)
        idx += 1
    cap.release()
    return frames, total_duration


# ══════════════════════════════════════════════════════════
# 場景分類器
# ══════════════════════════════════════════════════════════
class RuleSceneClassifier:
    """
    判斷規則（任一條件成立即判定為操作情境畫面）：
      1. 畫面中膚色像素比例 > 閾值（有真人出現）
      2. 邊緣密度高且畫面非過曝（複雜真實場景）
      3. 畫面文字／字幕中包含操作情境關鍵字（OCR 偵測）
    條件 3 僅在前兩個條件均未命中時執行，以節省 OCR 運算時間。
    """
    def is_usage_scene(self, frame_bgr: np.ndarray) -> bool:
        h, w = frame_bgr.shape[:2]
        total_pixels = h * w

        # ── 條件 1：膚色偵測（有真人）────────────────────────
        hsv  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
        skin = cv2.inRange(
            hsv,
            np.array([0, 30, 60],  np.uint8),
            np.array([20, 170, 255], np.uint8)
        )
        skin_ratio = np.count_nonzero(skin) / total_pixels
        if skin_ratio > RULE_SKIN_RATIO_THRESHOLD:
            return True

        # ── 條件 2：邊緣複雜度（複雜真實場景）───────────────
        gray  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        edge_density    = np.count_nonzero(edges) / total_pixels
        mean_brightness = gray.mean()
        if edge_density > RULE_EDGE_DENSITY_THRESHOLD and mean_brightness < RULE_BRIGHTNESS_MAX:
            return True

        # ── 條件 3：OCR 文字偵測（含操作情境關鍵字）─────────
        # 僅在前兩個條件均未命中時才執行，以降低運算負擔
        if _ocr_has_usage_keyword(frame_bgr):
            return True

        return False


class ClipSceneClassifier:
    def __init__(self):
        if not HAS_CLIP:
            raise ImportError('請先安裝 CLIP')
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model, self.preprocess = openai_clip.load('ViT-B/32', device=self.device)
        self.model.eval()
        all_prompts = USAGE_SCENE_PROMPTS + NON_USAGE_SCENE_PROMPTS
        tokens = openai_clip.tokenize(all_prompts).to(self.device)
        with torch.no_grad():
            self.text_features = self.model.encode_text(tokens)
            self.text_features /= self.text_features.norm(dim=-1, keepdim=True)
        self.n_usage = len(USAGE_SCENE_PROMPTS)

    def is_usage_scene(self, frame_bgr: np.ndarray) -> bool:
        from PIL import Image
        rgb   = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img_t = self.preprocess(Image.fromarray(rgb)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            feat  = self.model.encode_image(img_t)
            feat /= feat.norm(dim=-1, keepdim=True)
            probs = (feat @ self.text_features.T).squeeze(0).softmax(dim=-1).cpu().numpy()
        return bool(probs[:self.n_usage].sum() > probs[self.n_usage:].sum())


# ══════════════════════════════════════════════════════════
# 結果資料類別
# ══════════════════════════════════════════════════════════
@dataclass
class VideoFeatureResult:
    usage_scene_ratio: float
    total_duration:    float
    sampled_frames:    int
    usage_frames:      int
    classifier_mode:   str


# ══════════════════════════════════════════════════════════
# 單支影片分析
# ══════════════════════════════════════════════════════════
def analyze_video(video_path: str, classifier, interval_sec: float) -> VideoFeatureResult:
    frames, duration = sample_frames(video_path, interval_sec)
    usage_count = 0
    for i, frame in enumerate(frames):
        try:
            if classifier.is_usage_scene(frame):
                usage_count += 1
        except Exception as e:
            logger.warning(f'第 {i} 幀分類失敗: {e}')
    total = len(frames)
    ratio = round(usage_count / total, 4) if total > 0 else 0.0
    return VideoFeatureResult(
        usage_scene_ratio=ratio,
        total_duration=round(duration, 2),
        sampled_frames=total,
        usage_frames=usage_count,
        classifier_mode=CLASSIFIER_MODE,
    )


# ══════════════════════════════════════════════════════════
# 資料夾掃描：找出專案
# ══════════════════════════════════════════════════════════
def is_project_dir(path: str) -> bool:
    """含 *_content.txt 即視為專案資料夾。"""
    return any(f.endswith('_content.txt') for f in os.listdir(path))

def get_category(dirpath: str, root_dir: str) -> str:
    """擷取從根資料夾到專案之間的中間層路徑。"""
    rel   = os.path.relpath(dirpath, root_dir).replace('\\', '/')
    parts = rel.split('/')
    return '/'.join(parts[:-1]) if len(parts) > 1 else ''

# 支援的本地影片副檔名
VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.webm'}

def find_local_videos_folder(project_path: str) -> "Optional[str]":
    """尋找專案下的 *_videos/ 子資料夾，回傳路徑；找不到回傳 None。"""
    for item in os.listdir(project_path):
        if item.endswith('_videos') and os.path.isdir(os.path.join(project_path, item)):
            return os.path.join(project_path, item)
    return None

def read_log_video_entries(project_path: str) -> list:
    """讀取 *_resource_log.csv，回傳 type=Video 的所有列 {'filename', 'url'}。"""
    entries = []
    for fname in os.listdir(project_path):
        if not fname.endswith('_resource_log.csv'):
            continue
        csv_path = os.path.join(project_path, fname)
        with open(csv_path, 'r', encoding='utf-8-sig', errors='ignore') as f:
            reader = csv.DictReader(f)
            if not reader.fieldnames:
                continue
            fn_lower = {k.strip().lower(): k for k in reader.fieldnames}
            if 'type' not in fn_lower or 'url' not in fn_lower:
                continue
            type_col = fn_lower['type']
            url_col  = fn_lower['url']
            fn_col   = fn_lower.get('filename', None)
            for row in reader:
                if row[type_col].strip().lower() == 'video':
                    entries.append({
                        'filename': row[fn_col].strip() if fn_col else '',
                        'url':      row[url_col].strip(),
                    })
    return entries

def collect_video_entries(project_path: str) -> list:
    """
    影片來源優先順序：
      1. 本地 *_videos/ 資料夾（直接讀檔，每個影片檔為一筆）
      2. 若無本地資料夾或資料夾內無影片，改從 *_resource_log.csv 的 URL 下載

    回傳 [{'filename', 'local_path'(或None), 'url', 'source'}, ...]
    """
    videos_folder = find_local_videos_folder(project_path)

    # ── 優先：本地資料夾 ──────────────────────────────────
    if videos_folder:
        local_files = [
            f for f in os.listdir(videos_folder)
            if os.path.splitext(f)[1].lower() in VIDEO_EXTS
        ]
        if local_files:
            # 嘗試從 log 比對 URL（方便記錄，無則留空）
            log_entries = read_log_video_entries(project_path)
            log_url_map = {e['filename']: e['url'] for e in log_entries if e['filename']}
            return [
                {
                    'filename':   f,
                    'local_path': os.path.join(videos_folder, f),
                    'url':        log_url_map.get(f, ''),
                    'source':     'local',
                }
                for f in sorted(local_files)
            ]

    # ── 備援：從 log 取 URL ───────────────────────────────
    log_entries = read_log_video_entries(project_path)
    return [
        {
            'filename':   e['filename'],
            'local_path': None,
            'url':        e['url'],
            'source':     'url',
        }
        for e in log_entries if e['url']
    ]



# ══════════════════════════════════════════════════════════
# URL 統一下載入口  [Fix: resolve_url_to_video 補上定義]
# 支援：YouTube / Vimeo / 直連 mp4 / Google Drive / 其他平台
# ══════════════════════════════════════════════════════════
import urllib.request
import urllib.parse

def _is_direct_video_url(url: str) -> bool:
    """URL 結尾是影片副檔名 → 視為直連。"""
    path = urllib.parse.urlparse(url).path.lower()
    return any(path.endswith(ext) for ext in ('.mp4', '.mov', '.avi', '.webm', '.mkv'))


def _gdrive_direct_url(url: str) -> Optional[str]:
    """
    將 Google Drive 分享連結轉為直連下載 URL。
    支援格式：
      https://drive.google.com/file/d/{FILE_ID}/view
      https://drive.google.com/open?id={FILE_ID}
    """
    m = re.search(r'/file/d/([A-Za-z0-9_-]+)', url)
    if not m:
        m = re.search(r'[?&]id=([A-Za-z0-9_-]+)', url)
    if m:
        return f'https://drive.google.com/uc?export=download&id={m.group(1)}'
    return None


def _download_direct(url: str, output_dir: str) -> Optional[str]:
    """直連 URL（mp4 / Google Drive export）直接用 urllib 下載。"""
    try:
        parsed   = urllib.parse.urlparse(url)
        filename = os.path.basename(parsed.path) or 'video.mp4'
        if not os.path.splitext(filename)[1]:
            filename += '.mp4'
        dest = os.path.join(output_dir, filename)
        headers = {'User-Agent': 'Mozilla/5.0'}
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=120) as resp, \
             open(dest, 'wb') as out:
            out.write(resp.read())
        return dest if os.path.getsize(dest) > 1024 else None
    except Exception as e:
        logger.warning(f'直連下載失敗: {e}')
        return None


def resolve_url_to_video(url: str, output_dir: str) -> tuple[Optional[str], str]:
    """
    統一 URL 下載入口，回傳 (本地影片路徑, 下載方式說明)。
    失敗時回傳 (None, 失敗原因)。

    優先順序：
      1. YouTube / Shorts  → yt-dlp
      2. Vimeo / 其他平台  → yt-dlp（yt-dlp 支援 1000+ 平台）
      3. Google Drive      → urllib 直連 export URL
      4. 直連 mp4/mov/…   → urllib 直連
    """
    os.makedirs(output_dir, exist_ok=True)

    # ── 1. YouTube（含 Shorts）──────────────────────────────
    if extract_youtube_id(url):
        path = download_youtube_video(url, output_dir)
        if path:
            return path, 'yt-dlp/youtube'
        return None, 'yt-dlp/youtube 下載失敗（影片可能為私人或已下架）'

    # ── 2. Google Drive ─────────────────────────────────────
    if 'drive.google.com' in url:
        direct = _gdrive_direct_url(url)
        if direct:
            path = _download_direct(direct, output_dir)
            if path:
                return path, 'gdrive-direct'
        return None, 'Google Drive 下載失敗（請確認檔案為「任何人皆可檢視」）'

    # ── 3. 直連影片 URL ─────────────────────────────────────
    if _is_direct_video_url(url):
        path = _download_direct(url, output_dir)
        if path:
            return path, 'direct-url'
        return None, '直連 URL 下載失敗'

    # ── 4. 其他平台（Vimeo、Facebook 等）→ yt-dlp 嘗試 ─────
    path = download_youtube_video(url, output_dir)  # yt-dlp 通吃大多數平台
    if path:
        return path, 'yt-dlp/other'

    return None, f'所有下載方法均失敗（不支援的 URL 格式）: {url[:80]}'


print(f'✅ 核心函式載入完成（含 resolve_url_to_video）')
print(f'   OCR 支援：{"✅ 已啟用 (pytesseract)" if HAS_TESSERACT else "⚠️  未啟用（pytesseract 未安裝，將跳過文字偵測）"}')
print(f'   CLIP 支援：{"✅ 已啟用" if HAS_CLIP else "⚠️  未啟用（將使用規則式分類器）"}')


## ⑤ 執行批次分析並輸出 CSV

In [ ]:
# ── 初始化分類器 ──────────────────────────────────────────
if CLASSIFIER_MODE == 'clip' and HAS_CLIP:
    classifier = ClipSceneClassifier()
    print('✅ 使用 CLIP 分類器')
else:
    if CLASSIFIER_MODE == 'clip':
        print('⚠️  CLIP 未安裝，自動降級為規則式分類器')
    classifier = RuleSceneClassifier()
    print('✅ 使用規則式分類器')

os.makedirs(TMP_DIR, exist_ok=True)

# ── 輸出欄位 ──────────────────────────────────────────────
FIELDNAMES = [
    'project', 'category', 'video_filename', 'source',
    'url', 'download_method', 'usage_scene_ratio', 'total_duration',
    'sampled_frames', 'usage_frames', 'classifier_mode', 'status'
]

print('=' * 65)
print('     🎬 影片使用情境分析器（Colab 批次版）')
print('=' * 65)
print(f'\n🔍 掃描根資料夾：{ROOT_DIR}\n')

all_rows = []
project_count = 0
video_count   = 0

for dirpath, dirnames, filenames in os.walk(ROOT_DIR):
    if dirpath == ROOT_DIR:
        continue
    if not is_project_dir(dirpath):
        continue

    project_name = os.path.basename(dirpath)
    category     = get_category(dirpath, ROOT_DIR)
    project_count += 1

    print(f'\n📁 [{project_name}]  分類：{category}')

    video_entries = collect_video_entries(dirpath)

    if not video_entries:
        print(f'   ⚠️  找不到任何影片（本地及 URL 皆無），跳過')
        continue

    for entry in video_entries:
        filename   = entry['filename']
        local_path = entry['local_path']
        url        = entry['url']
        source     = entry['source']
        video_count += 1

        src_label = '📂 本地' if source == 'local' else '🌐 URL'
        print(f'   🎥  {src_label}  {filename or url[:55]}')

        row = {
            'project':         project_name,
            'category':        category,
            'video_filename':  filename,
            'source':          source,
            'url':             url,
            'classifier_mode': CLASSIFIER_MODE,
            'download_method': '',
        }

        try:
            if source == 'local' and local_path:
                # ── 本地影片：直接分析 ──────────────────────────────────
                video_path      = local_path
                download_method = 'local'
            else:
                # ── URL：統一走 resolve_url_to_video ───────────────────
                # 支援 YouTube、Vimeo、直連 mp4、Google Drive、其他平台
                video_path, download_method = resolve_url_to_video(url, TMP_DIR)
                if not video_path:
                    raise ValueError(
                        f'無法下載影片（方法：{download_method}）。'
                        f'請確認 URL 有效且為公開內容：{url[:80]}'
                    )
                print(f'       ⬇️  下載完成（{download_method}）')

            row['download_method'] = download_method
            result = analyze_video(video_path, classifier, SAMPLE_INTERVAL_SEC)

            row.update({
                'usage_scene_ratio': result.usage_scene_ratio,
                'total_duration':    result.total_duration,
                'sampled_frames':    result.sampled_frames,
                'usage_frames':      result.usage_frames,
                'status':            'ok',
            })
            print(f'       usage_scene_ratio={result.usage_scene_ratio:.2%}  '
                  f'時長={result.total_duration:.1f}s  '
                  f'分析幀={result.sampled_frames}')

        except Exception as e:
            row.update({
                'usage_scene_ratio': '',
                'total_duration':    '',
                'sampled_frames':    '',
                'usage_frames':      '',
                'status':            f'error: {e}',
            })
            print(f'       ❌ 失敗：{e}')

        all_rows.append(row)

# ── 輸出 CSV ──────────────────────────────────────────────
if all_rows:
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    with open(OUTPUT_CSV, 'a+', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(all_rows)
    print(f'\n✅ 完成！共分析 {project_count} 個專案、{video_count} 支影片')
    print(f'📁 結果已儲存至：{OUTPUT_CSV}')
else:
    print('\n⚠️  沒有找到任何影片資料。')


## ⑥ 預覽結果

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
print(f'共 {len(df)} 筆影片記錄')
df